# Setup

In [1]:
!pip install xarray tiler

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
# from osgeo import gdal
from tqdm import tqdm

%matplotlib inline

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference

# Config

In [5]:
# Data paths
INPUT_DIR = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/checkpoint_epoch_050.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs_meeting
Using device: cuda


# Load model

In [8]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, None, device=device) 
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=3,
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

Loading model from /explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


Encoder loaded with pretrained weights.


RuntimeError: Error(s) in loading state_dict for DINOSegmentation:
	size mismatch for decoder.up1.4.weight: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([512, 512, 3, 3]).
	size mismatch for decoder.up2.4.weight: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([256, 256, 3, 3]).
	size mismatch for decoder.up3.4.weight: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([128, 128, 3, 3]).
	size mismatch for decoder.up4.4.weight: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([64, 64, 3, 3]).

# Load data

## Load statistics from training to normalize inputs

In [ ]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
print(MEAN, STD)

# Inference

In [ ]:
# Run inference, create viz of inference
n_images = 3
images, predictions = run_datacube_inference(
    model=model,
    device=device,
    tiff_dir=INPUT_DIR,
    mean=MEAN,
    std=STD,
    output_path="sem_seg_inf_test_3_19.png",
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=0.25,
    threshold=0.75,
    normalize=True,
    save_inputs_dir=None,
    use_sliding=True,
)